# Informations

# 📑 Payments Transactional Report

## 🔗 Links
- **[BRD Title](https://cepalhellas.atlassian.net/wiki/spaces/DBTC/pages/263225346/BRD+-+Payments+Transactional+Report)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket](https://cepalhellas.atlassian.net/browse/DBTC-532)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket](https://cepalhellas.atlassian.net/browse/DBTC-958)**  
  A summary of findings from UAT.

# Authentication and Libraries import

In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback
import os

from typing import Any
 
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import col, when, lit, substring, concat_ws, coalesce, length, max, concat, format_string, expr, regexp_replace, cast, month, year, concat, left, expr, date_format, trim
 
from typing import Optional
 
import delta  # type: ignore
from pyspark.sql import SparkSession
 
from io import BytesIO
import pandas as pd
import json
import sys
 
from datetime import datetime
from pytz import timezone


from pyspark.sql.window import Window
from pyspark.sql.functions import row_number


"""
Accessing the Hive metastore requires specific considerations:
 
1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.
 
2. To access both Unity catalog and AWS services, an access token from Amazon is required.
 
3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html
 
4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.
 
5. After you authenticate successfully querying the Hive metastore is also working.
"""
 
try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared"
 
# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
   
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    
 
secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)
 
aws_key = aws_keys[0] if aws_keys else None
 
# Get AWS secret key from Databricks secrets
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
 
if mode == "shared":
    print("mode is shared")
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                          
else:
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1')
 
    S3_client = boto3.client('s3', region_name='eu-central-1')                          
 
 
# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']
 
# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

## Input Parameters
 
**report_args:** it is a json which contains all the appropriate parameters to run the report
 
`
{ "portfolio_name": "Mercury", "output_file_prefix": "Mercury_Payments_report",  "report_output": "/Volumes/qa-cdh-delivery/investorreports/Mercury/Payments/" }

 
 
**run_date:** is the date value to run the report. The run_date is empty when it is called by the orchestrator and that means "today"

In [0]:
athens_tz = timezone('Europe/Athens')
 
try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("\'", "\"")
except Exception as e:
    raise ValueError("Missing required input: report_args")
 
try:    
    report_name = dbutils.widgets.get("report_name")
except Exception as e:
    raise ValueError("Missing required input: report_name")
print(f"Report {report_name} is started")
 
try:
    run_date = dbutils.widgets.get("run_date")
    if run_date is None or run_date == "": raise ValueError("ERROR: run_date has no value. Remove the run_date completely if you want to run the job using the current date or add a value in the following format 'YYYY-MM-DD'")    
    run_date_str = run_date.replace('-', '')
    run_date = datetime.strptime(run_date, '%Y-%m-%d').date()
except Exception as e:
    run_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    run_date_str = datetime.now(athens_tz).strftime('%Y%m%d')
 
now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')
 
#Set the RefenceDate to be the previous working date in referal to run_Date 
referenceDate = run_date
# Step 1: Define the DF with the previous business day based on run_date
refdf = spark.sql(f"""
    SELECT PreviousBusinessDay 
    FROM `{env_prefix}-cdh`.reference.ref_calendar 
    WHERE CalendarDate = '{run_date}'
                """)  
# Extract the value into a Python variable
referenceDate = refdf.collect()[0]["PreviousBusinessDay"] if refdf.count() > 0 else None
print("Referenced Date:", referenceDate)
### End of RefenceDate setup
 
# Read JSON
json_data = json.loads(report_args)
 
# Access the values and store them into variables
# if "ssrs_workspace_id" not in json_data: raise KeyError("'ssrs_workspace_id' key is missing in the report_args JSON data.")
# if "ssrs_report_id" not in json_data: raise KeyError("'ssrs_report_id' key is missing in the report_args JSON data.")
if "portfolio_name" not in json_data: raise KeyError("'portfolio_name' key is missing in the report_args JSON data.")
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")
 
# workspace_id = "".join(json_data["ssrs_workspace_id"])
# ssrs_report_id = "".join(json_data["ssrs_report_id"])
portfolioName = json_data['portfolio_name']
output_file_prefix = "".join(json_data['output_file_prefix'])
report_output = json_data['report_output']
 
# print(f"Report {report_name} is running with the following arguments: report_args = {report_args}, report_name = {report_name}, portfolio_name = {portfolioName}, output_file_prefix = {output_file_prefix}, run_date = {run_date}")
 
# print(f"now is: {now_datetime_str}")

# Create Temp Views

In [0]:
dexposure = spark.sql(f"""
                    WITH all_exposures as (
                        SELECT CDI, AgreementKey, HasActiveSettlementFlag, AssetSegment, issecuredflag, SubServicer, ExposureKey, Bucket,
                        ROW_NUMBER() OVER (PARTITION BY ExposureKey ORDER BY BusinessDate DESC) as rn
                        FROM `{env_prefix}-cdh`.derivative.dexposure
                        WHERE PortfolioName LIKE ('%{portfolioName}%')
                    )
                    SELECT DISTINCT * FROM all_exposures WHERE rn = 1
                    """)
dexposure.createOrReplaceTempView("dexposure")


dparty = spark.sql(f"""
                   SELECT DISTINCT CDI, PartyType, TaxRegistrationNumber, FirstNameGR, LastNameGR, FirstNameENG, LastNameENG, GroupCode, GroupName, Segment 
                   FROM `{env_prefix}-cdh`.derivative.dparty
                   WHERE SourceSystem = 'profits' 
                   AND BusinessDate = '{referenceDate}' 
                   AND PortfolioName LIKE ('%{portfolioName}%')
                   """)
dparty.createOrReplaceTempView("dparty")


dcollection = spark.sql(f"""
                       SELECT * 
                       FROM `{env_prefix}-cdh`.derivative.dcollection 
                       WHERE BusinessDate = '{referenceDate}' 
                       AND PortfolioName LIKE ('%{portfolioName}%')
                       AND DATEDIFF(BusinessDate,Actualdate)<=365
                       """)
dcollection.createOrReplaceTempView("dcollection")

#defines secured perimeter for monza
servicerreportsecuredperimeter = spark.sql(f"""
                                WITH latest_servicer_report_securedperimeter as (
                                  SELECT max(ReferenceDate) as MaxReferenceDate
                                  FROM `{env_prefix}-reportinputfiles`.investor.servicer_report_securedperimeter
                                  WHERE ReferenceDate <= '{referenceDate}'
                              )
                                      SELECT
                                          DebtorCDI,
                                          CASE 
                                              WHEN SecUnsec = 'Unsecured D' THEN 'No'
                                              WHEN SecUnsec = 'Secured D' THEN 'Yes'
                                          END AS IsSecuredFlag,
                                          ReferenceDate
                                      FROM `{env_prefix}-reportinputfiles`.investor.servicer_report_securedperimeter sec_per
                                      JOIN latest_servicer_report_securedperimeter lsr 
                                          ON sec_per.ReferenceDate = lsr.MaxReferenceDate
                                      WHERE PortfolioName ILIKE '%monza%'
""")
servicerreportsecuredperimeter.createOrReplaceTempView("servicerreportsecuredperimeter")

# Code for selects

In [0]:
secured_perimeter_portfolios =['MONZA PORTFOLIO - D', 'MONZA PORTFOLIO - RETAIL', 'MONZA PORTFOLIO - WHOLESALE']

In [0]:
Investor = dcollection.alias("dc").join(
    dexposure.alias("de"),
    (
        col("de.ExposureKey") == col("dc.ExposureKey")
    ), "left"
 ).join(
     dparty.alias("dp"),
     col("dp.CDI") == coalesce(col("dc.PayerCDI"), col("dc.PrimaryOwnerCDI")), "left"
 ).join(
     servicerreportsecuredperimeter.alias("srp"),
     (col("srp.DebtorCDI") == col("dc.PrimaryOwnerCDI")) , "left"
).select(
    col("dc.DCollectionKey").alias("Collection Key"),  # ASK-5043: preserve payment uniqueness in .distinct() — TUN is NULL by design for adhoc Finance payments (EUCLID/Symbol/Pivot)
    col("dc.PortfolioName").alias("Portfolio Name"),
    col("dc.CollectionSourceKey").alias("Transaction unique id"),
    when(
    col("dp.partytype") == "Organization",
     col("dp.TaxRegistrationNumber")).otherwise(lit(None)).alias("Tax Number"), #DT-77 KS 30/6/2025
    coalesce(col("dc.PayerCDI"), col("dc.PrimaryOwnerCDI")).alias("CIF Number"),
    col("dc.PrimaryOwnerCDI").alias("CIF Number Primary Debtor"),
    col("de.AgreementKey").alias("AgreementNumber"),
    col("dc.Exposurekey").alias("Account Number"),
    col("dc.SourceExposurekey").alias("Host Account Number"),
    col("dc.actualdate").alias("Payment Date"),
    col("dc.recorddate").alias("Record Date"),
    col("dc.amount").cast("double").alias("Payment Amount"),
    when(
        col("dc.PaymentType") == "Είσπραξη από πελάτη, κατόπιν ρύθμισης που έγινε στη Cepal.",
            "Είσπραξη από πελάτη"
    ).otherwise(col("dc.PaymentType")).alias("PaymentType"),
    col("dc.currency").alias("Currency"),
    when(col("dc.FinalReason") == "Matched", "Yes").otherwise("No").alias("Is Payment Matched"),
    col("dc.MatchedDate").alias("Matched date"),
    when(
    col("dp.partytype") == "Organization",
        trim(concat_ws(" ", coalesce(col("dp.FirstNameGR"), lit("")), coalesce(col("dp.LastNameGR"), lit(""))))
    ).otherwise(lit(None)).alias("Party Name"), #DT-77 KS 30/6/2025
    when(
    col("dp.partytype") == "Organization",
        trim(concat_ws(" ", coalesce(col("dp.FirstNameENG"), lit("")), coalesce(col("dp.LastNameENG"), lit(""))))
     ).otherwise(lit(None)).alias("PartyNameENG"), #DT-77 KS 30/6/2025
    col("dp.GroupCode").alias("GroupID"),
    col("dp.GroupName").alias('GroupName'),
    col("dp.segment").alias("Customer Segment"),
    when(col("dc.PortfolioName").isin(secured_perimeter_portfolios), col("srp.IsSecuredFlag")) #secured perimeter for monza
    .otherwise(when(col("de.IsSecuredFlag") == True, "Yes").when(col("de.IsSecuredFlag") == False, "No"))
    .alias("Is Secured Flag"),
    col("de.SubServicer").alias("SubServicer")
).distinct()

# Fix to Null all Not Matched Recrds in the final table. Method to do this is to update the Final Data Frame based on Is Payment Matched Field
# List of columns to keep unchanged

columns_to_keep = [ "Collection Key", "Portfolio Name", "Transaction unique id",
    "CIF Number", "CIF Number Primary Debtor", "Account Number", "Host Account Number",
    "Payment Date", "Record Date", "PaymentType", "Currency",
    "Is Payment Matched", "Matched date", "Payment Amount"]

# Columns to nullify
columns_to_nullify = [c for c in Investor.columns if c not in columns_to_keep]

# Apply the update condition
for c in columns_to_nullify:
    Investor = Investor.withColumn(c, when(col("Is Payment Matched") == "No", lit(None)).otherwise(col(c)))

Investor = Investor.fillna("")

row_count = Investor.count()
print(f"The DataFrame has {row_count} rows.")

# Investor


In [0]:
import shutil
import os

Final_Investor = Investor.select(
    "Portfolio Name", 
    "Transaction unique id", 
    "Tax Number", 
    "CIF Number",
    "CIF Number Primary Debtor", 
    "AgreementNumber", 
    "Account Number", 
    "Host Account Number", 
    "Payment Date", 
    "Record Date", 
    "Payment Amount", 
    "PaymentType", 
    "Currency", 
    "Is Payment Matched", 
    "Matched date", 
    "Party Name", 
    "PartyNameEng", 
    "GroupID", 
    "GroupName", 
    "Customer Segment", 
    "Is Secured Flag", 
    "SubServicer"
)

print(f"The final DataFrame to be exported has {Final_Investor.count()} rows.")

# Get portfolio names from the DataFrame
portfolio_names = [row["Portfolio Name"] for row in Final_Investor.select("Portfolio Name").distinct().collect()]
portfolio_name_str = "_".join(portfolio_names).lower()


if Final_Investor.count() > 0 and "euclid" in portfolio_name_str.lower():
    
   # Define output directory path
    output_dir = f"{report_output}{run_date_str}/"
    os.makedirs(output_dir, exist_ok=True)

    # ------------------ 1. Export All EUCLID ------------------ #
    output_df_all = Final_Investor.toPandas()

    output_buffer_all = BytesIO()
    with pd.ExcelWriter(output_buffer_all) as writer:
        output_df_all.to_excel(writer, index=False)

    output_path_all = f"{output_dir}{output_file_prefix}_All_{referenceDate}.xlsx"

    with open(output_path_all, "wb") as f:
        f.write(output_buffer_all.getvalue())

    print(f"Full EUCLID data written to: {output_path_all}")

    # ------------------ 2. Export Specific SubServicer ------------------ #
    specific_df = Final_Investor.filter(
        col("SubServicer") == "EUCLID DV01 - ASSIGN FOR SERVICING"
    ).toPandas()

    output_buffer_specific = BytesIO()
    with pd.ExcelWriter(output_buffer_specific) as writer:
        specific_df.to_excel(writer, index=False)

    output_path_specific = f"{output_dir}{output_file_prefix}_Specific_{referenceDate}.xlsx"

    with open(output_path_specific, "wb") as f:
        f.write(output_buffer_specific.getvalue())

    print(f"Specific EUCLID SubServicer data written to: {output_path_specific}")
else:
    
    output_df = Final_Investor.toPandas()

    output_buffer = BytesIO()
    with pd.ExcelWriter(output_buffer) as writer:
        output_df.to_excel(writer, index=False)
    
    #Define the output directory path
    output_dir = f"{report_output}{run_date_str}/"

    # Ensure the directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Define the output file path
    output_path = f"{output_dir}{output_file_prefix}_Investor_{referenceDate}.xlsx"

    with open(output_path, "wb") as f:
        f.write(output_buffer.getvalue())
    
    print(f"Results successfully written to Volume at: {output_path}")